In [ ]:
import tkinter as tk
import random
import heapq
import time
import math

CELL_SIZE = 25

# ---------------- NODE CLASS ----------------
class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0
        self.h = 0
        self.f = 0

    def __lt__(self, other):
        # Tie-breaker: prefer larger g to reduce heap size
        if self.f == other.f:
            return self.g > other.g
        return self.f < other.f

# ---------------- MAIN APP ----------------
class PathfindingApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Dynamic Pathfinding Agent")

        self.rows = 20
        self.cols = 20
        self.grid = []
        self.start = (0, 0)
        self.goal = (19, 19)
        self.agent_position = self.start
        self.dynamic_mode = False

        self.algorithm = tk.StringVar(value="A*")
        self.heuristic_type = tk.StringVar(value="Manhattan")

        self.create_controls()
        self.create_grid()

    # ---------- GUI CONTROLS ----------
    def create_controls(self):
        frame = tk.Frame(self.root)
        frame.pack()

        tk.Label(frame, text="Rows").grid(row=0, column=0)
        self.row_entry = tk.Entry(frame, width=5)
        self.row_entry.insert(0, "20")
        self.row_entry.grid(row=0, column=1)

        tk.Label(frame, text="Cols").grid(row=0, column=2)
        self.col_entry = tk.Entry(frame, width=5)
        self.col_entry.insert(0, "20")
        self.col_entry.grid(row=0, column=3)

        tk.Button(frame, text="Create Grid", command=self.reset_grid).grid(row=0, column=4)
        tk.Button(frame, text="Random Maze", command=self.generate_random).grid(row=0, column=5)
        tk.Checkbutton(frame, text="Dynamic Mode", command=self.toggle_dynamic).grid(row=0, column=6)

        tk.Label(frame, text="Algorithm").grid(row=1, column=0)
        tk.OptionMenu(frame, self.algorithm, "A*", "GBFS").grid(row=1, column=1)

        tk.Label(frame, text="Heuristic").grid(row=1, column=2)
        tk.OptionMenu(frame, self.heuristic_type, "Manhattan", "Euclidean").grid(row=1, column=3)

        tk.Button(frame, text="Start Search", command=self.start_search).grid(row=1, column=4)

        # Metrics
        self.metrics_label = tk.Label(frame, text="")
        self.metrics_label.grid(row=2, column=0, columnspan=7)

    # ---------- GRID CREATION ----------
    def create_grid(self):
        self.canvas = tk.Canvas(self.root, width=self.cols * CELL_SIZE, height=self.rows * CELL_SIZE)
        self.canvas.pack()
        self.grid = [[0 for _ in range(self.cols)] for _ in range(self.rows)]
        self.draw_grid()

    def reset_grid(self):
        self.rows = int(self.row_entry.get())
        self.cols = int(self.col_entry.get())
        self.goal = (self.rows - 1, self.cols - 1)
        self.agent_position = self.start

        self.canvas.destroy()
        self.create_grid()

    def draw_grid(self):
        self.canvas.delete("all")
        for r in range(self.rows):
            for c in range(self.cols):
                color = "white"
                if self.grid[r][c] == 1:
                    color = "black"
                if (r, c) == self.start:
                    color = "blue"
                if (r, c) == self.goal:
                    color = "purple"
                if (r, c) == self.agent_position:
                    color = "orange"

                self.canvas.create_rectangle(
                    c * CELL_SIZE,
                    r * CELL_SIZE,
                    (c + 1) * CELL_SIZE,
                    (r + 1) * CELL_SIZE,
                    fill=color,
                    outline="gray"
                )

        self.canvas.bind("<Button-1>", self.toggle_wall)

    def toggle_wall(self, event):
        col = event.x // CELL_SIZE
        row = event.y // CELL_SIZE
        if (row, col) != self.start and (row, col) != self.goal:
            self.grid[row][col] = 1 - self.grid[row][col]
        self.draw_grid()

    def generate_random(self):
        density = 0.3
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) != self.start and (r, c) != self.goal:
                    self.grid[r][c] = 1 if random.random() < density else 0
        self.draw_grid()

    def toggle_dynamic(self):
        self.dynamic_mode = not self.dynamic_mode

    # ---------- HEURISTICS ----------
    def heuristic(self, a, b):
        if self.heuristic_type.get() == "Manhattan":
            return abs(a[0] - b[0]) + abs(a[1] - b[1])
        else:
            # Use Euclidean squared for speed & stability
            return (a[0] - b[0])**2 + (a[1] - b[1])**2

    # ---------- SEARCH ----------
    def search(self):
        start_time = time.time()
        open_list = []
        visited = {}
        start_node = Node(self.agent_position)
        heapq.heappush(open_list, start_node)
        nodes_visited = 0

        while open_list:
            current = heapq.heappop(open_list)
            nodes_visited += 1

            if current.position == self.goal:
                path = []
                while current:
                    path.append(current.position)
                    current = current.parent
                end_time = time.time()
                return path[::-1], nodes_visited, (end_time - start_time) * 1000

            # mark visited with f-value to avoid re-expansion
            if current.position in visited and visited[current.position] <= current.f:
                continue
            visited[current.position] = current.f

            for dx, dy in [(0,1),(1,0),(0,-1),(-1,0)]:
                r = current.position[0] + dx
                c = current.position[1] + dy
                if 0 <= r < self.rows and 0 <= c < self.cols:
                    if self.grid[r][c] == 1:
                        continue
                    neighbor = Node((r,c), current)
                    if self.algorithm.get() == "A*":
                        neighbor.g = current.g + 1
                        neighbor.h = self.heuristic(neighbor.position, self.goal)
                        neighbor.f = neighbor.g + neighbor.h
                    else:
                        neighbor.h = self.heuristic(neighbor.position, self.goal)
                        neighbor.f = neighbor.h
                    heapq.heappush(open_list, neighbor)

        return None, nodes_visited, 0

    # ---------- SEARCH EXECUTION ----------
    def start_search(self):
        self.agent_position = self.start
        self.draw_grid()
        # compute full path first
        path, nodes, exec_time = self.search()
        if not path:
            self.metrics_label.config(text="No Path Found!")
            return
        # animate safely
        self.animate_path(path, nodes, exec_time)

    # ---------- ANIMATION ----------
    def animate_path(self, path, nodes, exec_time):
        for step in path:
            if self.dynamic_mode and random.random() < 0.05:
                self.spawn_obstacle()

            # if next step blocked, recompute path from current position
            if self.grid[step[0]][step[1]] == 1:
                self.agent_position = step
                new_path, nodes, exec_time = self.search()
                if not new_path:
                    self.metrics_label.config(text="Blocked!")
                    return
                path = new_path
                return self.animate_path(path, nodes, exec_time)

            self.agent_position = step

            # draw only current agent
            self.canvas.delete("agent")
            self.canvas.create_rectangle(
                step[1]*CELL_SIZE,
                step[0]*CELL_SIZE,
                (step[1]+1)*CELL_SIZE,
                (step[0]+1)*CELL_SIZE,
                fill="orange",
                outline="gray",
                tags="agent"
            )
            self.root.update()
            time.sleep(0.05)  # smaller delay, faster animation

        # draw final path in green
        for r, c in path:
            if (r, c) != self.start and (r, c) != self.goal:
                self.canvas.create_rectangle(
                    c*CELL_SIZE,
                    r*CELL_SIZE,
                    (c+1)*CELL_SIZE,
                    (r+1)*CELL_SIZE,
                    fill="green",
                    outline="gray"
                )

        # redraw start and goal
        sr, sc = self.start
        gr, gc = self.goal
        self.canvas.create_rectangle(sc*CELL_SIZE, sr*CELL_SIZE, (sc+1)*CELL_SIZE, (sr+1)*CELL_SIZE, fill="blue", outline="gray")
        self.canvas.create_rectangle(gc*CELL_SIZE, gr*CELL_SIZE, (gc+1)*CELL_SIZE, (gr+1)*CELL_SIZE, fill="purple", outline="gray")

        self.metrics_label.config(
            text=f"Nodes Visited: {nodes} | Path Cost: {len(path)-1} | Time: {exec_time:.2f} ms"
        )

    # ---------- DYNAMIC OBSTACLE ----------
    def spawn_obstacle(self):
        free_cells = [(r,c) for r in range(self.rows)
                      for c in range(self.cols)
                      if self.grid[r][c] == 0
                      and (r,c) != self.start
                      and (r,c) != self.goal]
        if free_cells:
            r,c = random.choice(free_cells)
            self.grid[r][c] = 1

# ---------- RUN APP ----------
root = tk.Tk()
app = PathfindingApp(root)
root.mainloop()